# DeepSeek-R1-0528-Qwen3-8B — C Coding Fine-Tuning Experiment

**Hardware**: Tesla T4 16 GB  
**Goal**: Compare QLoRA fine-tuning via HuggingFace Transformers vs Unsloth, with and without DeepSpeed, on a synthetic C coding instruction dataset.

## Experiment Flow
1. Package imports
2. Dataset inspection
3. Baseline evaluation (pre fine-tuning)
4. Fine-tuning with HF Transformers (no DeepSpeed)
5. Fine-tuning with HF Transformers + DeepSpeed ZeRO-2
6. Fine-tuning with Unsloth
7. Post fine-tuning evaluation for all three checkpoints
8. Side-by-side metrics comparison table and chart

## 1. Package Imports
All packages needed for the experiment. Install order is documented in `requirements.txt`.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────
import json
import os
import subprocess
import sys
import tempfile
import time

# ── PyTorch ───────────────────────────────────────────────────────────────
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM (GB)       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")

# ── HuggingFace ecosystem ──────────────────────────────────────────────────
import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
import peft
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
import datasets
from datasets import Dataset
import trl
from trl import SFTTrainer
import accelerate
import bitsandbytes

# ── Evaluation ────────────────────────────────────────────────────────────
import evaluate as hf_evaluate
import sacrebleu

# ── Data / Visualisation ──────────────────────────────────────────────────
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── Utilities ─────────────────────────────────────────────────────────────
from IPython.display import display

print("\nPackage versions:")
for pkg_name, pkg in [
    ("transformers", transformers),
    ("peft",         peft),
    ("datasets",     datasets),
    ("trl",          trl),
    ("accelerate",   accelerate),
    ("bitsandbytes", bitsandbytes),
    ("matplotlib",   matplotlib),
]:
    print(f"  {pkg_name:<14}: {pkg.__version__}")

## 2. Dataset Inspection
Load `data/c_coding_dataset.json` and examine the synthetic C coding instruction dataset.

In [ ]:
DATASET_PATH = "data/c_coding_dataset.json"

with open(DATASET_PATH) as f:
    raw_data = json.load(f)

train_samples = raw_data["train"]
eval_samples  = raw_data["eval"]

print(f"Train samples : {len(train_samples)}")
print(f"Eval  samples : {len(eval_samples)}")
print("\n--- Example train sample ---")
ex = train_samples[0]
print(f"Instruction: {ex['instruction']}")
print(f"Output:\n{ex['output'][:300]}...")

In [ ]:
# Domain distribution
domain_keywords = {
    "Pointer": ["pointer", "ptr", "dereference", "address"],
    "Memory": ["malloc", "free", "memory", "heap", "allocat", "realloc"],
    "Algorithm": ["sort", "search", "algorithm", "binary", "linked list", "stack", "queue", "recursi"],
    "Struct": ["struct", "typedef", "member"],
    "File I/O": ["file", "fopen", "fclose", "fread", "fwrite", "fprintf", "fscanf"],
}

domain_counts = {d: 0 for d in domain_keywords}
for s in train_samples + eval_samples:
    instr = s["instruction"].lower()
    for domain, keywords in domain_keywords.items():
        if any(k in instr for k in keywords):
            domain_counts[domain] += 1
            break

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(domain_counts.keys(), domain_counts.values(), color=sns.color_palette("tab10"))
ax.set_title("Dataset domain distribution")
ax.set_ylabel("Sample count")
plt.tight_layout()
plt.show()
print(domain_counts)

## 3. Baseline Evaluation (pre fine-tuning)
Run `evaluate.py` against the base model to record Pass@1, BLEU-4, and Exact Match.

> **Note**: This cell calls `evaluate.py` as a subprocess so it runs in a clean process with its own GPU context. This avoids VRAM conflicts with the training cells below.

In [ ]:
MODEL_NAME = "deepseek-ai/DeepSeek-R1-0528-Qwen3-8B"  # change to local path if downloaded

def run_evaluate(model_path: str, output_file: str) -> dict:
    """Run evaluate.py and return the parsed metrics dict."""
    cmd = [
        sys.executable, "evaluate.py",
        "--model_path", model_path,
        "--dataset_path", DATASET_PATH,
        "--output", output_file,
        "--split", "eval",
    ]
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
    with open(output_file) as f:
        return json.load(f)

baseline_metrics = run_evaluate(MODEL_NAME, "results_baseline.json")
print("\nBaseline metrics:", json.dumps(baseline_metrics, indent=2))

## 4. Fine-Tuning — HF Transformers (no DeepSpeed)
QLoRA via HuggingFace Transformers + PEFT. Single GPU, paged AdamW 8-bit.

In [ ]:
ft_no_ds_cmd = [
    sys.executable, "finetune_transformers.py",
    "--model_name", MODEL_NAME,
    "--dataset_path", DATASET_PATH,
    "--output_dir", "finetuned-transformers",
    "--max_seq_length", "512",
    # no --use_deepspeed
]

print("Starting HF Transformers fine-tuning (no DeepSpeed)...")
t0 = time.time()
result = subprocess.run(ft_no_ds_cmd, capture_output=True, text=True)
elapsed = time.time() - t0
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
print(f"\nCompleted in {elapsed:.1f}s")

# Read run_meta.json written by finetune_transformers.py
with open("finetuned-transformers/run_meta.json") as f:
    ft_no_ds_meta = json.load(f)
print(json.dumps(ft_no_ds_meta, indent=2))

## 5. Fine-Tuning — HF Transformers + DeepSpeed ZeRO-2
Same QLoRA config, but launched through `deepspeed` for ZeRO-2 optimizer + gradient partitioning.

In [ ]:
ft_ds_cmd = [
    "deepspeed", "--num_gpus=1", "finetune_transformers.py",
    "--model_name", MODEL_NAME,
    "--dataset_path", DATASET_PATH,
    "--output_dir", "finetuned-transformers-ds",
    "--max_seq_length", "512",
    "--use_deepspeed",
    "--ds_config", "configs/ds_config.json",
]

print("Starting HF Transformers fine-tuning WITH DeepSpeed ZeRO-2...")
t0 = time.time()
result = subprocess.run(ft_ds_cmd, capture_output=True, text=True)
elapsed = time.time() - t0
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
print(f"\nCompleted in {elapsed:.1f}s")

with open("finetuned-transformers-ds/run_meta.json") as f:
    ft_ds_meta = json.load(f)
print(json.dumps(ft_ds_meta, indent=2))

## 6. Fine-Tuning — Unsloth
Uses `unsloth.FastLanguageModel` with patched attention kernels. Same LoRA config, no DeepSpeed needed.

In [ ]:
ft_unsloth_cmd = [
    sys.executable, "finetune_unsloth.py",
    "--model_name", MODEL_NAME,
    "--dataset_path", DATASET_PATH,
    "--output_dir", "finetuned-unsloth",
    "--max_seq_length", "512",
]

print("Starting Unsloth fine-tuning...")
t0 = time.time()
result = subprocess.run(ft_unsloth_cmd, capture_output=True, text=True)
elapsed = time.time() - t0
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
print(f"\nCompleted in {elapsed:.1f}s")

with open("finetuned-unsloth/run_meta.json") as f:
    ft_unsloth_meta = json.load(f)
print(json.dumps(ft_unsloth_meta, indent=2))

## 7. Post Fine-Tuning Evaluation
Run `evaluate.py` on all three checkpoints and collect metrics.

In [ ]:
checkpoints = {
    "Transformers (no DS)": ("finetuned-transformers",    "results_ft_transformers.json"),
    "Transformers + DS":    ("finetuned-transformers-ds",  "results_ft_ds.json"),
    "Unsloth":              ("finetuned-unsloth",          "results_ft_unsloth.json"),
}

post_metrics = {}
for label, (ckpt_dir, out_file) in checkpoints.items():
    if os.path.isdir(ckpt_dir):
        print(f"\n=== Evaluating: {label} ===")
        post_metrics[label] = run_evaluate(ckpt_dir, out_file)
    else:
        print(f"[SKIP] {ckpt_dir} not found — run the fine-tuning cells first")

## 8. Metrics Comparison Table & Chart
Side-by-side comparison of Pass@1, BLEU-4, and Exact Match across all runs.

In [ ]:
# ── Build comparison DataFrame ─────────────────────────────────────────────
all_results = {"Baseline": baseline_metrics, **post_metrics}

rows = []
for label, m in all_results.items():
    rows.append({
        "Run":         label,
        "Pass@1":      m.get("pass_at_1", float("nan")),
        "BLEU-4":      m.get("bleu4",     float("nan")),
        "Exact Match": m.get("exact_match", float("nan")),
    })

df_metrics = pd.DataFrame(rows).set_index("Run")
print("=== Evaluation Metrics ===")
display(df_metrics.style.format("{:.4f}").background_gradient(cmap="RdYlGn", axis=0))

In [ ]:
# ── Training performance comparison ───────────────────────────────────────
meta_sources = {
    "Transformers (no DS)": ft_no_ds_meta if 'ft_no_ds_meta' in dir() else {},
    "Transformers + DS":    ft_ds_meta    if 'ft_ds_meta'    in dir() else {},
    "Unsloth":              ft_unsloth_meta if 'ft_unsloth_meta' in dir() else {},
}

perf_rows = []
for label, meta in meta_sources.items():
    perf_rows.append({
        "Run":                 label,
        "Time (s)":            meta.get("elapsed_seconds",  float("nan")),
        "Peak GPU Mem (MB)":   meta.get("peak_gpu_mem_mb",  float("nan")),
        "Tokens/sec (est.)": meta.get("tokens_per_sec",   float("nan")),
    })

df_perf = pd.DataFrame(perf_rows).set_index("Run")
print("=== Training Performance ===")
display(df_perf.style.format("{:.1f}").background_gradient(cmap="RdYlGn_r", axis=0))

In [ ]:
# ── Bar chart: evaluation metrics ─────────────────────────────────────────
metrics_to_plot = ["Pass@1", "BLEU-4", "Exact Match"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = sns.color_palette("tab10", n_colors=len(df_metrics))

for ax, metric in zip(axes, metrics_to_plot):
    vals = df_metrics[metric]
    bars = ax.bar(vals.index, vals.values, color=colors)
    ax.set_title(metric, fontsize=13)
    ax.set_ylim(0, max(vals.max() * 1.2, 0.1))
    ax.set_xticklabels(vals.index, rotation=20, ha="right", fontsize=9)
    for bar, v in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{v:.3f}", ha="center", va="bottom", fontsize=8)

fig.suptitle("Pre vs Post Fine-Tuning — Evaluation Metrics", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to metrics_comparison.png")

In [ ]:
# ── Bar chart: training efficiency ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
perf_cols = ["Time (s)", "Peak GPU Mem (MB)"]
colors2 = sns.color_palette("Set2", n_colors=len(df_perf))

for ax, col in zip(axes, perf_cols):
    vals = df_perf[col].dropna()
    bars = ax.bar(vals.index, vals.values, color=colors2[:len(vals)])
    ax.set_title(col, fontsize=12)
    ax.set_xticklabels(vals.index, rotation=20, ha="right", fontsize=9)
    for bar, v in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{v:.0f}", ha="center", va="bottom", fontsize=9)

fig.suptitle("Training Efficiency — Transformers vs Transformers+DS vs Unsloth", fontsize=13)
plt.tight_layout()
plt.savefig("efficiency_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to efficiency_comparison.png")

## Summary

| | Baseline | Transformers | Transformers+DS | Unsloth |
|---|---|---|---|---|
| Pass@1 | — | — | — | — |
| BLEU-4 | — | — | — | — |
| Exact Match | — | — | — | — |
| Training Time (s) | N/A | — | — | — |
| Peak GPU Mem (MB) | N/A | — | — | — |

*Fill in after running all cells.*

### Key Takeaways
- **Unsloth vs Transformers**: Unsloth typically achieves ~1.5–2× faster training and ~30–50% lower peak GPU memory through kernel-level optimisations.
- **DeepSpeed ZeRO-2**: Reduces per-GPU memory for optimizer states and gradients; on a single T4 the benefit is modest but allows larger effective batch sizes.
- **Fine-tuning delta**: Even 3 epochs on 50 samples should show measurable improvement in compilation rate (Pass@1) for the instruction-following format.